In [1]:
import os
import csv
from pymongo import MongoClient
import certifi
import pandas as pd

MONGO_URI = os.getenv("MONGO_URI")
client = MongoClient(MONGO_URI, tlsCAFile=certifi.where())
db = client['geogame_db']
leaderboard = db['leaderboard']

all_scores = list(leaderboard.find())

filename = "geopuzzle_data.csv"
    
headers = ["date", "name", "guesses", "time", "streak", "ip_address"]
    
df = pd.DataFrame(all_scores)

if not df.empty:
    # Filter out the MongoDB '_id' and arrange the columns exactly how you want them
    headers = ["date", "name", "guesses", "time", "streak", "ip_address"]
    
    # This ensures we only grab columns that actually exist, preventing errors if a column is missing
    existing_headers = [col for col in headers if col in df.columns]
    df = df[existing_headers]

# Display the first 5 rows in your notebook
df.head()

,date,name,guesses,time,streak,ip_address
0,2026-04-18,Ben,3,15.44,NaN,NaN
1,2026-04-19,Freja,6,37.59,NaN,NaN
2,2026-04-19,Tyler,8,100.26,NaN,NaN
3,2026-04-19,Grant,3,37.07,NaN,NaN
4,2026-04-20,Ben,3,22.55,NaN,NaN


In [2]:
from IPython.display import display

display(df)

# df.to_csv('my_data.csv')

,date,name,guesses,time,streak,ip_address
0,2026-04-18,Ben,3,15.44,NaN,NaN
1,2026-04-19,Freja,6,37.59,NaN,NaN
2,2026-04-19,Tyler,8,100.26,NaN,NaN
3,2026-04-19,Grant,3,37.07,NaN,NaN
4,2026-04-20,Ben,3,22.55,NaN,NaN
...,...,...,...,...,...,...
115,2026-05-07,Ben,4,29.27,9.0,86.15.161.90
116,2026-05-07,Tyler = goat,2,9.41,8.0,86.15.161.90
117,2026-05-07,Lexie,6,67.09,5.0,86.18.55.236
118,2026-05-07,soph,5,60.30,8.0,86.15.175.232


In [4]:
df = df.sort_values(by=['date', 'time', 'guesses'])

# 2. Assign the official daily rank to EVERYONE (1st, 2nd, 3rd...)
df['official_position'] = df.groupby('date').cumcount() + 1

# 3. Filter the dataframe to only include rows where the name contains "tyler"
tyler_df = df[df['name'].str.contains('tyler', case=False, na=False)]

# 4. Group by date to merge multiple attempts on the same day
daily_summary = tyler_df.groupby('date').agg(
    avg_position=('official_position', 'mean'),
    avg_time=('time', 'mean'),
    daily_wins=('official_position', lambda x: 1 if (x == 1).any() else 0),
    total_attempts=('name', 'count') # Added this so you can see how many times he played!
).reset_index()

# 5. Calculate the Grand Totals
total_wins = daily_summary['daily_wins'].sum()

# We calculate the overall averages using the raw tyler_df so every attempt is weighted equally
overall_avg_position = tyler_df['official_position'].mean()
overall_avg_time = tyler_df['time'].mean()

# 6. Display the final analytics
print(f"👑 Tyler's Total Wins: {total_wins}")
print(f"📊 Tyler's Overall Avg Finish: {overall_avg_position:.1f}")
print(f"⏱️ Tyler's Overall Avg Time: {overall_avg_time:.1f}s\n")

print("📅 Tyler's Daily Breakdown:")
# Format the floats so they look clean in the console (e.g., 2.5 instead of 2.500000)
daily_summary['avg_position'] = daily_summary['avg_position'].round(1)
daily_summary['avg_time'] = daily_summary['avg_time'].round(1)

print(daily_summary.to_string(index=False))

👑 Tyler's Total Wins: 5
📊 Tyler's Overall Avg Finish: 2.5
⏱️ Tyler's Overall Avg Time: 44.5s

📅 Tyler's Daily Breakdown:
      date  avg_position  avg_time  daily_wins  total_attempts
2026-04-19           3.0     100.3           0               1
2026-04-20           4.0      32.5           0               1
2026-04-21           1.0      25.1           1               1
2026-04-22           3.0     132.4           0               1
2026-04-23           2.0      42.8           0               1
2026-04-24           4.0      52.8           0               1
2026-04-25           1.0      15.1           1               1
2026-04-26           2.0      59.4           0               1
2026-04-27           2.0      56.1           0               1
2026-04-28           2.0       6.6           0               1
2026-04-29           6.0      47.9           0               1
2026-04-30           3.0      35.7           0               1
2026-05-01           3.0      97.1           0              

In [7]:
import pandas as pd

# 1. Sort everyone by date, then time, then guesses
df = df.sort_values(by=['date', 'time', 'guesses'])

# 2. Assign the official daily rank to EVERYONE
df['official_position'] = df.groupby('date').cumcount() + 1

# 3. Filter the dataframe for "ben" and save it to a properly named variable
ben_df = df[df['name'].str.contains('ben', case=False, na=False)]

# 4. Group by date to merge multiple attempts
daily_summary = ben_df.groupby('date').agg(
    avg_position=('official_position', 'mean'),
    avg_time=('time', 'mean'),
    daily_wins=('official_position', lambda x: 1 if (x == 1).any() else 0),
    total_attempts=('name', 'count')
).reset_index()

# 5. Calculate the Grand Totals
total_wins = daily_summary['daily_wins'].sum()

overall_avg_position = ben_df['official_position'].mean()
overall_avg_time = ben_df['time'].mean()

# 6. Display the final analytics
print(f"👑 Ben's Total Wins: {total_wins}")
print(f"📊 Ben's Overall Avg Finish: {overall_avg_position:.1f}")
print(f"⏱️ Ben's Overall Avg Time: {overall_avg_time:.1f}s\n")

print("📅 Ben's Daily Breakdown:")
daily_summary['avg_position'] = daily_summary['avg_position'].round(1)
daily_summary['avg_time'] = daily_summary['avg_time'].round(1)

print(daily_summary.to_string(index=False))

👑 Ben's Total Wins: 7
📊 Ben's Overall Avg Finish: 2.3
⏱️ Ben's Overall Avg Time: 40.3s

📅 Ben's Daily Breakdown:
      date  avg_position  avg_time  daily_wins  total_attempts
2026-04-18           1.0      15.4           1               1
2026-04-20           3.0      22.6           0               1
2026-04-21           3.0      64.2           0               1
2026-04-22           2.0      91.4           0               1
2026-04-23           1.0      42.8           1               1
2026-04-24           1.0       9.8           1               1
2026-04-25           2.0      29.4           0               1
2026-04-26           5.0     120.8           0               1
2026-04-27           4.0     106.5           0               1
2026-04-28           4.0      19.1           0               1
2026-04-29           5.0      47.8           0               1
2026-04-30           1.0       9.7           1               1
2026-05-01           1.0      32.1           1               1
2026-